In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

In [ ]:
# =====================================================
# PASO 1: CARGAR Y PREPARAR DATOS
# =====================================================

print("=" * 60)
print("PIPELINE COMPLETO: IMPUTACIÓN Y PREDICCIÓN DE DIABETES")
print("=" * 60)

# Cargar dataset preparado de fase anterior
df = pd.read_parquet("nhanes_limpio.parquet")
print(f"Dataset original: {df.shape}")

# Variable objetivo
le_target = LabelEncoder()
y = le_target.fit_transform(df["Diabetes"])
X = df.drop(columns=["Diabetes"])

# División estratificada en train/test
X_train, X_test, y_train_arr, y_test_arr = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Convertir y_train e y_test a Series con los mismos índices que X_train y X_test
y_train = pd.Series(y_train_arr, index=X_train.index)
y_test = pd.Series(y_test_arr, index=X_test.index)

# Separar variables categóricas y numéricas
categorical_vars = X.select_dtypes(include='category').columns.tolist()
numerical_vars = X.select_dtypes(include=["float64", "int32"]).columns.tolist()

print(f"Variables categóricas ({len(categorical_vars)}): {categorical_vars}")
print(f"Variables numéricas ({len(numerical_vars)}): {numerical_vars}")

In [ ]:
# =====================================================
# FUNCIONES AUXILIARES
# =====================================================

from sklearn.base import clone

# Función 1: codificación de categóricas después de imputar
def encode_categorical_variables(X_train, X_test, categorical_vars):
    X_train_encoded = X_train.copy()
    X_test_encoded = X_test.copy()
    encoders = {}

    for var in categorical_vars:
        le = LabelEncoder()
        X_train_encoded[var] = le.fit_transform(X_train_encoded[var].astype(str))
        encoders[var] = le

        test_vals = X_test_encoded[var].astype(str)
        test_encoded = []
        for val in test_vals:
            if val in le.classes_:
                test_encoded.append(le.transform([val])[0])
            else:
                most_frequent = X_train[var].mode()[0] if not X_train[var].empty else le.classes_[0]
                test_encoded.append(le.transform([str(most_frequent)])[0])

        X_test_encoded[var] = test_encoded

    return X_train_encoded, X_test_encoded, encoders


# Función 2: imputación por método (categóricas SIEMPRE con moda)
def apply_imputation_method(X_train, X_test, method='simple_mean', categorical_vars=[], numerical_vars=[]):
    X_train_imp = X_train.copy()
    X_test_imp = X_test.copy()

    # Imputación para categóricas: siempre con moda (por seguridad y coherencia)
    if categorical_vars:
        imputer_cat = SimpleImputer(strategy='most_frequent')
        X_train_imp[categorical_vars] = imputer_cat.fit_transform(X_train_imp[categorical_vars])
        X_test_imp[categorical_vars] = imputer_cat.transform(X_test_imp[categorical_vars])

    # Imputación de numéricas
    if method == 'simple_mean':
        imputer_num = SimpleImputer(strategy='mean')

    elif method == 'simple_median':
        imputer_num = SimpleImputer(strategy='median')

    elif method == 'knn':
        imputer_num = KNNImputer(n_neighbors=5)

    elif method == 'iterative':
        imputer_num = IterativeImputer(estimator=RandomForestRegressor(n_estimators=10, max_depth=5, n_jobs=-1, random_state=42),
                                       random_state=42, max_iter=10, tol=1e-1, skip_complete=True)

    else:
        raise ValueError(f"Método de imputación no reconocido: {method}")

    if numerical_vars:
        X_train_imp[numerical_vars] = imputer_num.fit_transform(X_train_imp[numerical_vars])
        X_test_imp[numerical_vars] = imputer_num.transform(X_test_imp[numerical_vars])

    return X_train_imp, X_test_imp


# Función 3: entrenamiento y evaluación (sin cambios)
def train_and_evaluate(X_train, X_test, y_train, y_test, method_name, balance_method='none'):
    print(f"\n--- MÉTODO: {method_name.upper()} | BALANCE: {balance_method.upper()} ---")

    X_train_bal = X_train.copy()
    y_train_bal = y_train.copy()

    if balance_method == 'smote':
        smote = SMOTE(random_state=42)
        X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
        print(f"Después de SMOTE: {np.bincount(y_train_bal)}")

    elif balance_method == 'undersample':
        rus = RandomUnderSampler(random_state=42)
        X_train_bal, y_train_bal = rus.fit_resample(X_train, y_train)
        print(f"Después de Undersampling: {np.bincount(y_train_bal)}")

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_bal)
    X_test_scaled = scaler.transform(X_test)

    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'KNN': KNeighborsClassifier(n_neighbors=5)
    }

    results = {}

    for model_name, model in models.items():
        if balance_method == 'none' and hasattr(model, 'class_weight'):
            model.set_params(class_weight='balanced')

        model.fit(X_train_scaled, y_train_bal)
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]

        auc = roc_auc_score(y_test, y_proba)
        f1 = f1_score(y_test, y_pred)
        pr_auc = average_precision_score(y_test, y_proba)
        cv_auc = cross_val_score(model, X_train_scaled, y_train_bal,
                                 cv=StratifiedKFold(5, shuffle=True, random_state=42),
                                 scoring='roc_auc')

        results[model_name] = {
            'AUC_test': auc,
            'AUC_cv_mean': cv_auc.mean(),
            'AUC_cv_std': cv_auc.std(),
            'F1': f1,
            'PR_AUC': pr_auc
        }

        print(f"{model_name}:")
        print(f"  AUC Test: {auc:.4f}")
        print(f"  AUC CV: {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")
        print(f"  F1: {f1:.4f}")
        print(f"  PR AUC: {pr_auc:.4f}")

    return results


In [ ]:
all_results = {}

imp_method = 'simple_mean'
for balance_method in ['none', 'smote']:
    print(f"\n🔄 APLICANDO IMPUTACIÓN: {imp_method.upper()} + {balance_method.upper()}")

    # Imputar (categóricas siempre con moda, numéricas según método)
    X_train_imp, X_test_imp = apply_imputation_method(
        X_train, X_test, imp_method, categorical_vars, numerical_vars
    )

    # Codificar categóricas después de imputar
    X_train_imp, X_test_imp, _ = encode_categorical_variables(
        X_train_imp, X_test_imp, categorical_vars
    )

    # Entrenar y evaluar
    key = f"{imp_method}_{balance_method}"
    all_results[key] = train_and_evaluate(
        X_train_imp, X_test_imp, y_train, y_test, imp_method, balance_method
    )


In [ ]:
imp_method = 'simple_median'
for balance_method in ['none', 'smote']:
    print(f"\n🔄 APLICANDO IMPUTACIÓN: {imp_method.upper()} + {balance_method.upper()}")

    # Imputación segura
    X_train_imp, X_test_imp = apply_imputation_method(
        X_train, X_test, imp_method, categorical_vars, numerical_vars
    )

    # Codificación segura (después de imputar)
    X_train_imp, X_test_imp, _ = encode_categorical_variables(
        X_train_imp, X_test_imp, categorical_vars
    )

    # Entrenamiento y evaluación
    key = f"{imp_method}_{balance_method}"
    all_results[key] = train_and_evaluate(
        X_train_imp, X_test_imp, y_train, y_test, imp_method, balance_method
    )


In [ ]:
imp_method = 'knn'
for balance_method in ['none', 'smote']:
    print(f"\n🔄 APLICANDO IMPUTACIÓN: {imp_method.upper()} + {balance_method.upper()}")

    # Imputación: numéricas con KNN, categóricas con moda
    X_train_imp, X_test_imp = apply_imputation_method(
        X_train, X_test, imp_method, categorical_vars, numerical_vars
    )

    # Codificación de categóricas tras imputación
    X_train_imp, X_test_imp, _ = encode_categorical_variables(
        X_train_imp, X_test_imp, categorical_vars
    )

    # Entrenamiento y evaluación
    key = f"{imp_method}_{balance_method}"
    all_results[key] = train_and_evaluate(
        X_train_imp, X_test_imp, y_train, y_test, imp_method, balance_method
    )


In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

imp_method = 'iterative'

for balance_method in ['none', 'smote']:
    print(f"\n🔄 APLICANDO IMPUTACIÓN: {imp_method.upper()} + {balance_method.upper()}")

    # Imputación segura
    X_train_imp, X_test_imp = apply_imputation_method(
        X_train, X_test, imp_method, categorical_vars, numerical_vars
    )

    # Codificación tras imputación
    X_train_imp, X_test_imp, _ = encode_categorical_variables(
        X_train_imp, X_test_imp, categorical_vars
    )

    # Evaluación
    key = f"{imp_method}_{balance_method}"
    all_results[key] = train_and_evaluate(
        X_train_imp, X_test_imp, y_train, y_test, imp_method, balance_method
    )



In [ ]:
from sklearn.preprocessing import LabelEncoder

# Copias de seguridad con eliminación de filas con NA
X_train_complete = X_train.dropna().copy()
X_test_complete = X_test.dropna().copy()
y_train_complete = y_train.loc[X_train_complete.index]
y_test_complete = y_test.loc[X_test_complete.index]

# Codificación categórica segura (LabelEncoder con control de valores no vistos)
le_dict = {}
for col in categorical_vars:
    le = LabelEncoder()
    X_train_complete[col] = le.fit_transform(X_train_complete[col].astype(str))
    le_dict[col] = le

    # Codificar test de forma segura
    test_vals = X_test_complete[col].astype(str)
    test_encoded = []
    for val in test_vals:
        if val in le.classes_:
            test_encoded.append(le.transform([val])[0])
        else:
            most_frequent = X_train_complete[col].mode()[0]
            test_encoded.append(le.transform([str(most_frequent)])[0])
    X_test_complete[col] = test_encoded

# Escalado (antes de SMOTE para mantener coherencia con el resto del experimento)
scaler = StandardScaler()
X_train_complete[numerical_vars] = scaler.fit_transform(X_train_complete[numerical_vars])
X_test_complete[numerical_vars] = scaler.transform(X_test_complete[numerical_vars])

# Aplicar SMOTE al conjunto completo
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_complete, y_train_complete)


In [ ]:
print("\n" + "="*60)
print("EXPERIMENTO BASELINE: CASOS COMPLETOS (SIN IMPUTAR)")
print("="*60)

baseline_results = {}

for version, X_tr, y_tr in [
    ("none", X_train_complete, y_train_complete),
    ("smote", X_train_smote, y_train_smote)
]:
    print(f"\n--- DROPNA | BALANCE: {version.upper()} ---")

    for name, model in zip(
        ["Logistic Regression", "Random Forest", "KNN"],
        [LogisticRegression(max_iter=1000, random_state=42),
         RandomForestClassifier(n_estimators=100, random_state=42),
         KNeighborsClassifier()]
    ):
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_test_complete)
        y_proba = model.predict_proba(X_test_complete)[:, 1]

        auc_test = roc_auc_score(y_test_complete, y_proba)
        f1 = f1_score(y_test_complete, y_pred)
        pr_auc = average_precision_score(y_test_complete, y_proba)

        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        auc_cv_scores = cross_val_score(model, X_tr, y_tr, scoring='roc_auc', cv=cv)
        auc_cv_mean = auc_cv_scores.mean()
        auc_cv_std = auc_cv_scores.std()

        print(f"{name}:\n  AUC Test: {auc_test:.4f}\n  AUC CV: {auc_cv_mean:.4f} ± {auc_cv_std:.4f}\n  F1: {f1:.4f}\n  PR AUC: {pr_auc:.4f}")

        # Guardar resultados en all_results (coherente con el resto del experimento)
        key = f"dropna_{version}"
        if key not in all_results:
            all_results[key] = {}
        all_results[key][name] = {
            'AUC_test': auc_test,
            'AUC_cv_mean': auc_cv_mean,
            'AUC_cv_std': auc_cv_std,
            'F1': f1,
            'PR_AUC': pr_auc
        }


In [ ]:
# Ver cuántas filas quedan tras aplicar dropna en X
X_train_complete = X_train.dropna()
X_test_complete = X_test.dropna()

print("\n" + "="*60)
print("FILAS RESTANTES DESPUÉS DE DROPNA (CASOS COMPLETOS)")
print("="*60)
print(f"Train original: {X_train.shape[0]}  →  Después de dropna: {X_train_complete.shape[0]}")
print(f"Test original:  {X_test.shape[0]}   →  Después de dropna: {X_test_complete.shape[0]}")


In [ ]:
print("\n" + "="*60)
print("RESUMEN FINAL DE RESULTADOS (ORDENADO POR F1)")
print("="*60)

# Recopilar resultados
results_summary = []

# Métodos de imputación evaluados
imputation_methods = ['simple_mean', 'simple_median', 'knn', 'iterative', 'dropna']

# Recoger todos los resultados desde all_results
for exp_key, models_results in all_results.items():
    parts = exp_key.split('_')
    balance_method = parts[-1]
    imp_method = '_'.join(parts[:-1])

    for model_name, metrics in models_results.items():
        results_summary.append({
            'Imputación': imp_method,
            'Balance': balance_method,
            'Modelo': model_name,
            'AUC_Test': metrics['AUC_test'],
            'AUC_CV_Mean': metrics['AUC_cv_mean'],
            'F1': metrics['F1'],
            'PR_AUC': metrics['PR_AUC']
        })

# Crear DataFrame y ordenar por F1
results_df = pd.DataFrame(results_summary)
results_df = results_df.sort_values(by='F1', ascending=False)

# Mostrar top 10 por F1
print("\n🔝 TOP 10 COMBINACIONES POR F1")
display(results_df.head(10).round(4))

# Mostrar mejor resultado por método de imputación según F1
print("\n" + "-"*50)
print("MEJOR RESULTADO POR MÉTODO DE IMPUTACIÓN (según F1)")
print("-"*50)

for method in imputation_methods:
    method_results = results_df[results_df['Imputación'] == method]
    if not method_results.empty:
        best = method_results.loc[method_results['F1'].idxmax()]
        print(f"{method.upper()}: {best['F1']:.4f} ({best['Modelo']} + {best['Balance']})")

# Guardar si quieres exportarlo
# results_df.to_csv("resultados_experimento_1_F1.csv", index=False)

print(f"\n✅ Experimentos completados.")
print(f"📊 Total combinaciones evaluadas: {len(results_df)}")
print(f"🏆 Mejor F1 alcanzado: {results_df['F1'].max():.4f}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Crear copia para orden
df_plot = results_df.copy()
df_plot["Etiqueta"] = df_plot.apply(
    lambda row: f"{row['Imputación']}_{row['Balance']} - {row['Modelo']}", axis=1
)

# Ordenar por F1
df_plot = df_plot.sort_values(by="F1", ascending=False)

plt.figure(figsize=(14, 8))
sns.barplot(data=df_plot, y="Etiqueta", x="F1", hue="Etiqueta", palette="viridis", legend=False)
plt.title("Comparativa de F1-score por combinación de método + modelo + balance")
plt.xlabel("F1-score")
plt.ylabel("Método de imputación y modelo")
plt.tight_layout()
plt.show()



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Crear matriz resumen de F1-score por imputación y modelo
pivot_f1 = results_df.pivot_table(
    index="Imputación",
    columns="Modelo",
    values="F1",
    aggfunc="max"  # mejor F1 por modelo/imputación
)

# Reordenar las filas para más claridad
orden_filas = ['dropna', 'simple_mean', 'simple_median', 'knn', 'iterative']
pivot_f1 = pivot_f1.reindex(orden_filas)

# Heatmap
plt.figure(figsize=(8, 5))
sns.heatmap(pivot_f1, annot=True, fmt=".3f", cmap="YlGnBu", cbar_kws={"label": "F1-score"})
plt.title("F1-score máximo por modelo y método de imputación")
plt.ylabel("Método de imputación")
plt.xlabel("Modelo de clasificación")
plt.tight_layout()
plt.show()


In [ ]:
# Crear matriz resumen de AUC Test por imputación y modelo
pivot_auc = results_df.pivot_table(
    index="Imputación",
    columns="Modelo",
    values="AUC_Test",
    aggfunc="max"
).reindex(orden_filas)

# Crear matriz resumen de PR AUC por imputación y modelo
pivot_pr_auc = results_df.pivot_table(
    index="Imputación",
    columns="Modelo",
    values="PR_AUC",
    aggfunc="max"
).reindex(orden_filas)

# Gráfico 1: AUC Test
plt.figure(figsize=(8, 5))
sns.heatmap(pivot_auc, annot=True, fmt=".3f", cmap="Blues", cbar_kws={"label": "AUC Test"})
plt.title("AUC Test máximo por modelo y método de imputación")
plt.ylabel("Método de imputación")
plt.xlabel("Modelo de clasificación")
plt.tight_layout()
plt.show()

# Gráfico 2: PR AUC
plt.figure(figsize=(8, 5))
sns.heatmap(pivot_pr_auc, annot=True, fmt=".3f", cmap="Oranges", cbar_kws={"label": "PR AUC"})
plt.title("Área bajo la curva Precision-Recall (PR AUC) por imputación y modelo")
plt.ylabel("Método de imputación")
plt.xlabel("Modelo de clasificación")
plt.tight_layout()
plt.show()
